# Notebook 5 (Fase 1.5) — Backtest de Estrategia SMA → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Calcular un backtest real y determinístico de una estrategia de cruce de medias móviles (**SMA-20 / SMA-50**) usando los indicadores reales ya calculados y guardados por el Notebook 1 en la colección `precios_ohlcv`, y persistir los resultados (operaciones, curva de equity y métricas) en MongoDB.

**Importante:** No se usa ningún número aleatorio (`random`, `Math.random()`, etc.). Cada señal de compra/venta, cada operación, la curva de equity y las métricas finales se derivan matemáticamente del histórico real de precios de Yahoo Finance (via `yfinance`, ingestado por el Notebook 1).

**Nota técnica — por qué NO se usa `vectorbt`:** en Google Colab, `vectorbt` exige `pandas>=3.0` y `numba>=0.66`, versiones que rompen el entorno base de Colab (que trae `pandas==2.2.2`) y producen conflictos binarios irresolubles entre `numpy`/`pandas`/`numba` (`ImportError` al importar la librería). Por eso el motor de backtest de este notebook se implementó **manualmente con pandas y numpy** —ya preinstalados y estables en Colab— siguiendo exactamente la misma lógica de simulación de una cartera con señales de entrada/salida (motor tipo *vectorized backtesting*), sin depender de librerías externas frágiles.

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada con SMA-20 y SMA-50).

**Salida:** Colección `resultados_backtest` poblada con, para cada uno de los 5 tickers: la lista de operaciones (trades), la curva de equity y las métricas de desempeño (retorno total, Sharpe, máximo drawdown, win rate, número de operaciones).

*Nota:* Este notebook NO levanta ninguna API — solo calcula el backtest y lo guarda en MongoDB. La exposición vía FastAPI (endpoint `/api/backtest/{'{'}ticker{'}'}`, colección `resultados_backtest`) se implementará en un notebook aparte, siguiendo el mismo patrón del Notebook 3.

**Pipeline:** MongoDB (`precios_ohlcv`, datos reales) → señales SMA-20/SMA-50 → motor de backtest (pandas/numpy) → **MongoDB (`resultados_backtest`)**

In [1]:
# Paso 1 — Instalar librerias necesarias
# NOTA IMPORTANTE: NO se usa la libreria 'vectorbt'. En Google Colab, vectorbt
# exige pandas>=3.0 y numba>=0.66, versiones que rompen el entorno base de
# Colab (que trae pandas==2.2.2) y generan conflictos irresolubles entre
# numpy/pandas/numba (ImportError al importar vectorbt). Para evitar ese
# problema, el backtest se calcula manualmente con pandas y numpy —ya
# preinstalados y estables en Colab— siguiendo exactamente la misma logica
# de cruce de medias moviles (100% deterministica, sin numeros aleatorios).
!pip install "pymongo[srv]" --quiet

print("Librerias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.0 MB/s eta 0:00:00
Librerias instaladas correctamente


In [2]:
# Paso 2 — Conectar a MongoDB Atlas
# La contrasena NUNCA se escribe en texto plano en el notebook: se pide de forma
# oculta con getpass y se inserta en la URI de conexion en tiempo de ejecucion.
# Mismo patron de conexion usado en los Notebooks 1, 2, 3 y 4.
from getpass import getpass
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd
import numpy as np

DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi("1"))

# Verificar la conexion con un ping explicito, manejando errores
try:
    client.admin.command("ping")
    print("Conectado a MongoDB Atlas correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

Password de MongoDB Atlas: ··········
Conectado a MongoDB Atlas correctamente


In [3]:
# Paso 3 — Cargar datos reales desde MongoDB (coleccion precios_ohlcv del Notebook 1)
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]

def cargar_datos(ticker):
    """Lee de MongoDB los precios OHLCV + indicadores (SMA-20, SMA-50, etc.) de un
    ticker, ordenados cronologicamente. Estos son datos reales de Yahoo Finance
    ingestados por el Notebook 1 — no se descarga ni se inventa nada aqui.
    """
    docs = list(db["precios_ohlcv"].find({"ticker": ticker}).sort("fecha", 1))
    df = pd.DataFrame(docs)
    if not df.empty:
        df["fecha"] = pd.to_datetime(df["fecha"])
        df = df.set_index("fecha")
    return df

df_prueba = cargar_datos("BVN")
print(f"BVN: {len(df_prueba)} dias reales cargados desde MongoDB")
print(df_prueba[["close", "sma_20", "sma_50"]].tail())

BVN: 251 dias reales cargados desde MongoDB
            close   sma_20   sma_50
fecha                              
2026-07-09  29.55  31.1165  32.8038
2026-07-10  30.00  31.0825  32.7818
2026-07-13  29.82  30.9355  32.7778
2026-07-14  31.03  30.8160  32.7466
2026-07-15  30.71  30.6085  32.7120


In [4]:
# Paso 4 — Estrategia deterministica: cruce de medias moviles SMA-20 / SMA-50
# Regla 100% mecanica sobre los indicadores reales guardados en MongoDB:
#   ENTRADA (BUY):  cuando sma_20 cruza HACIA ARRIBA a sma_50  (cruce dorado)
#   SALIDA  (SELL): cuando sma_20 cruza HACIA ABAJO a sma_50   (cruce de la muerte)
# No hay ningun componente aleatorio: la señal depende unicamente de los
# valores reales de sma_20 y sma_50 calculados en el Notebook 1.

def generar_senales(df):
    """Genera las señales booleanas de entrada y salida a partir del cruce SMA-20/SMA-50."""
    df = df.dropna(subset=["close", "sma_20", "sma_50"]).copy()

    por_encima = df["sma_20"] > df["sma_50"]
    por_encima_ayer = por_encima.shift(1).fillna(False)

    entradas = por_encima & (~por_encima_ayer)   # cruce hacia arriba (golden cross)
    salidas = (~por_encima) & por_encima_ayer    # cruce hacia abajo (death cross)

    return df, entradas, salidas

df_bvn, entradas_bvn, salidas_bvn = generar_senales(df_prueba)
print(f"BVN: {entradas_bvn.sum()} señales de entrada, {salidas_bvn.sum()} señales de salida")

BVN: 3 señales de entrada, 3 señales de salida


/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)


In [5]:
# Paso 5 — Motor de backtest manual (pandas/numpy) sobre precios reales
# Simula una cartera "todo o nada": al recibir una señal de ENTRADA se invierte
# TODO el efectivo disponible (menos comision); al recibir una señal de SALIDA
# se liquida TODA la posicion (menos comision). Es una logica de backtest
# vectorizado equivalente a la de VectorBT, pero implementada a mano para no
# depender de una libreria externa fragil en Colab (ver nota tecnica arriba).

CAPITAL_INICIAL = 10_000.0   # capital inicial simulado en USD
COMISION = 0.001              # 0.1% de comision por operacion, tipico de un broker retail

def ejecutar_backtest(df, entradas, salidas, capital_inicial=CAPITAL_INICIAL, comision=COMISION):
    """Recorre la serie de precios reales dia a dia aplicando las señales de
    entrada/salida y devuelve la curva de equity y la lista de operaciones cerradas.
    """
    cash = capital_inicial
    acciones = 0.0
    posicion_abierta = False
    fecha_entrada = None
    precio_entrada = None
    costo_entrada_neto = None

    equity_curve = []
    trades = []

    for fecha, fila in df.iterrows():
        precio = float(fila["close"])

        # --- Señal de ENTRADA: se invierte todo el efectivo disponible ---
        if bool(entradas.loc[fecha]) and not posicion_abierta and cash > 0:
            monto_comision = cash * comision
            cash_neto = cash - monto_comision
            acciones = cash_neto / precio
            costo_entrada_neto = cash_neto
            cash = 0.0
            posicion_abierta = True
            fecha_entrada = fecha
            precio_entrada = precio

        # --- Señal de SALIDA: se liquida toda la posicion ---
        elif bool(salidas.loc[fecha]) and posicion_abierta:
            valor_bruto = acciones * precio
            monto_comision = valor_bruto * comision
            valor_neto = valor_bruto - monto_comision

            ganancia_perdida = valor_neto - costo_entrada_neto
            retorno_pct = ganancia_perdida / costo_entrada_neto if costo_entrada_neto else None

            trades.append({
                "fecha_entrada": str(fecha_entrada.date()),
                "fecha_salida": str(fecha.date()),
                "precio_entrada": precio_entrada,
                "precio_salida": precio,
                "ganancia_perdida": ganancia_perdida,
                "retorno_pct": retorno_pct,
            })

            cash = valor_neto
            acciones = 0.0
            posicion_abierta = False
            fecha_entrada = None
            precio_entrada = None
            costo_entrada_neto = None

        # Valor de la cartera al cierre del dia (en efectivo o en acciones, segun corresponda)
        valor_portafolio = cash + acciones * precio
        equity_curve.append((fecha, valor_portafolio))

    return {"equity_curve": equity_curve, "trades": trades}

df_bvn, entradas_bvn, salidas_bvn = generar_senales(df_prueba)
resultado_bvn = ejecutar_backtest(df_bvn, entradas_bvn, salidas_bvn)
valor_final_bvn = resultado_bvn["equity_curve"][-1][1] if resultado_bvn["equity_curve"] else CAPITAL_INICIAL

print("Backtest de prueba (BVN) ejecutado con el motor manual")
print(f"  Retorno total: {(valor_final_bvn / CAPITAL_INICIAL - 1):.2%}")
print(f"  Valor final del portafolio: ${valor_final_bvn:,.2f}")
print(f"  Operaciones cerradas: {len(resultado_bvn['trades'])}")

Backtest de prueba (BVN) ejecutado con el motor manual
  Retorno total: 19.81%
  Valor final del portafolio: $11,980.58
  Operaciones cerradas: 3


/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)


In [6]:
# Paso 6 — Extraer metricas, operaciones y curva de equity; guardar en MongoDB
from datetime import datetime

def limpiar(v):
    """Convierte NaN/inf y tipos numpy a tipos nativos, validos para JSON/MongoDB."""
    if v is None:
        return None
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if np.isnan(v) or np.isinf(v):
        return None
    return round(v, 4)

def extraer_metricas(resultado):
    """Calcula las metricas clave de desempeño a partir de la curva de equity y las operaciones."""
    equity_curve = resultado["equity_curve"]
    trades = resultado["trades"]

    valores = np.array([v for _, v in equity_curve], dtype=float)
    capital_final = float(valores[-1]) if len(valores) else CAPITAL_INICIAL
    retorno_total = (capital_final / CAPITAL_INICIAL) - 1

    # Sharpe ratio anualizado (252 dias de trading), con retornos diarios simples
    if len(valores) > 1:
        retornos_diarios = np.diff(valores) / valores[:-1]
        std = retornos_diarios.std()
        sharpe = (retornos_diarios.mean() / std) * np.sqrt(252) if std > 0 else None
    else:
        sharpe = None

    # Maximo drawdown: mayor caida porcentual desde un maximo historico de la curva
    if len(valores) > 0:
        maximo_acumulado = np.maximum.accumulate(valores)
        drawdowns = (valores - maximo_acumulado) / maximo_acumulado
        max_drawdown = float(drawdowns.min())
    else:
        max_drawdown = None

    total_trades = len(trades)
    if total_trades > 0:
        ganadoras = sum(1 for t in trades if t["ganancia_perdida"] > 0)
        win_rate = ganadoras / total_trades
    else:
        win_rate = None

    metricas = {
        "retorno_total": limpiar(retorno_total),
        "sharpe_ratio": limpiar(sharpe),
        "max_drawdown": limpiar(max_drawdown),
        "win_rate": limpiar(win_rate),
        "num_operaciones": total_trades,
        "capital_inicial": CAPITAL_INICIAL,
        "capital_final": limpiar(capital_final),
    }
    return metricas

def extraer_operaciones(resultado):
    """Limpia cada operacion (trade) real cerrada por el motor de backtest, lista para MongoDB."""
    operaciones = []
    for t in resultado["trades"]:
        operaciones.append({
            "fecha_entrada": t["fecha_entrada"],
            "fecha_salida": t["fecha_salida"],
            "precio_entrada": limpiar(t["precio_entrada"]),
            "precio_salida": limpiar(t["precio_salida"]),
            "ganancia_perdida": limpiar(t["ganancia_perdida"]),
            "retorno_pct": limpiar(t["retorno_pct"]),
        })
    return operaciones

def extraer_curva_equity(resultado):
    """Serializa la curva de equity (valor del portafolio dia a dia) para graficarla en el frontend."""
    curva = [
        {"fecha": str(fecha.date()), "valor": limpiar(valor)}
        for fecha, valor in resultado["equity_curve"]
    ]
    return curva

def procesar_ticker_backtest(ticker):
    """Pipeline completo de backtest para un ticker: carga -> señales -> motor manual -> MongoDB."""
    try:
        df = cargar_datos(ticker)
        if df.empty or len(df) < 60:
            print(f"  [{ticker}] Datos insuficientes para backtest, se omite.")
            return

        df, entradas, salidas = generar_senales(df)
        if entradas.sum() == 0:
            print(f"  [{ticker}] Sin señales de cruce SMA en el periodo disponible, se omite.")
            return

        resultado = ejecutar_backtest(df, entradas, salidas)
        metricas = extraer_metricas(resultado)
        operaciones = extraer_operaciones(resultado)
        curva_equity = extraer_curva_equity(resultado)

        # Se borra el backtest previo del mismo ticker/estrategia para evitar duplicados al re-ejecutar
        db["resultados_backtest"].delete_many({"ticker": ticker, "estrategia": "SMA_20_50"})
        db["resultados_backtest"].insert_one({
            "ticker": ticker,
            "estrategia": "SMA_20_50",
            **metricas,
            "operaciones": operaciones,
            "curva_equity": curva_equity,
            "fecha_backtest": datetime.now()
        })

        rt = metricas["retorno_total"]
        nop = metricas["num_operaciones"]
        print(f"  [{ticker}] retorno_total={rt:.2%} | operaciones={nop} | guardado en MongoDB" if rt is not None
              else f"  [{ticker}] backtest guardado ({nop} operaciones)")

    except Exception as e:
        print(f"  [{ticker}] Error durante el backtest: {e}")

print("Funcion procesar_ticker_backtest lista")

Funcion procesar_ticker_backtest lista


In [7]:
# Paso 7 — Ejecutar el backtest para los 5 tickers del proyecto
print("Ejecutando backtest SMA-20/SMA-50 (motor manual pandas/numpy) para los 5 tickers...")
print()

for ticker in TICKERS:
    procesar_ticker_backtest(ticker)

print()
print("Backtest completo!")

Ejecutando backtest SMA-20/SMA-50 (motor manual pandas/numpy) para los 5 tickers...



/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)


  [FSM] retorno_total=-2.55% | operaciones=2 | guardado en MongoDB
  [VOLCABC1.LM] retorno_total=42.51% | operaciones=2 | guardado en MongoDB


/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)
/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)


  [ABX.TO] retorno_total=22.77% | operaciones=3 | guardado en MongoDB
  [BVN] retorno_total=19.81% | operaciones=3 | guardado en MongoDB


/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)
/tmp/ipykernel_614/50383579.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  por_encima_ayer = por_encima.shift(1).fillna(False)


  [BHP] retorno_total=12.14% | operaciones=3 | guardado en MongoDB

Backtest completo!


In [8]:
# Paso 8 — Celda de verificacion final: resultados guardados en MongoDB
print("Resumen de la coleccion resultados_backtest:")
print()

tickers_con_backtest = set()
for doc in db["resultados_backtest"].find({"estrategia": "SMA_20_50"}):
    t = doc["ticker"]
    tickers_con_backtest.add(t)
    rt = doc.get("retorno_total")
    dd = doc.get("max_drawdown")
    nop = doc.get("num_operaciones")
    rt_str = f"{rt:.2%}" if rt is not None else "N/A"
    dd_str = f"{dd:.2%}" if dd is not None else "N/A"
    print(f"  [OK] {t}: retorno_total={rt_str} | max_drawdown={dd_str} | operaciones={nop}")

print()
faltantes = set(TICKERS) - tickers_con_backtest
if not faltantes:
    print("Verificacion exitosa: los 5 tickers tienen backtest guardado en MongoDB.")
else:
    print(f"Atencion: no se genero backtest para: {sorted(faltantes)} (posiblemente sin señales de cruce SMA aun).")

Resumen de la coleccion resultados_backtest:

  [OK] FSM: retorno_total=-2.55% | max_drawdown=-37.26% | operaciones=2
  [OK] VOLCABC1.LM: retorno_total=42.51% | max_drawdown=-29.64% | operaciones=2
  [OK] ABX.TO: retorno_total=22.77% | max_drawdown=-20.75% | operaciones=3
  [OK] BVN: retorno_total=19.81% | max_drawdown=-33.62% | operaciones=3
  [OK] BHP: retorno_total=12.14% | max_drawdown=-19.80% | operaciones=3

Verificacion exitosa: los 5 tickers tienen backtest guardado en MongoDB.


## Resultado

La colección `resultados_backtest` contiene, para cada ticker, el resultado real y determinístico de la estrategia de cruce **SMA-20/SMA-50**: la lista completa de operaciones (fecha de entrada/salida, precios, ganancia/pérdida), la curva de equity día a día, y las métricas de desempeño (retorno total, Sharpe, máximo drawdown, win rate, número de operaciones).

Ningún valor de esta colección es aleatorio: todo se deriva de los precios reales de Yahoo Finance ingestados por el Notebook 1 y de la ejecución determinística del motor de backtest sobre esos datos.

**Pendiente (fuera de este notebook):** exponer esta colección con un nuevo endpoint FastAPI, p. ej. `GET /api/backtest/{ticker}`, siguiendo el mismo patrón del Notebook 3, para que los paneles de backtest del frontend dejen de mostrar el mensaje de "aún no tiene datos reales" y se conecten igual que el resto del dashboard.

**Pipeline:** MongoDB (`precios_ohlcv`) → cruce SMA-20/SMA-50 → motor de backtest (pandas/numpy) → **MongoDB (`resultados_backtest`)** ✓